In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
df=spark.table('workspace.bronze.dwh_loc_a101')
df.display()

## **trim string feild**

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df=df.withColumn(field.name,trim(col(field.name)))

**Standardize Country**

In [0]:
mapping = {"US": "USA","United States":"USA","Canada":"UK","United Kingdom":"UK","DE":"Germany"} 

df = df.replace(mapping, subset=["CNTRY"])
df=df.withColumn("CNTRY",when((col("CNTRY").isNull()),"N/A").when(col("CNTRY")=="",'N/A').otherwise(col("CNTRY")))

## **clean CID**

In [0]:

df=df.withColumn('CID',regexp_replace(col('CID'),'-',''))

## **Renaming Columns**

In [0]:
rename={'CID':'customer_number','CNTRY':'country'}

for old,new in rename.items():
    df=df.withColumnRenamed(old,new)
df.display()

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('workspace.silver.erp_location')